<a href="https://colab.research.google.com/github/Koji-Accounting-Tech/D365-Finance-Data-Validator/blob/main/D365_Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. Excelファイルの読み込み
df = pd.read_excel('test_data.xlsx')

# 2. データの先頭5行を表示して中身を確認
print("--- データの先頭5行 ---")
display(df.head())

# 3. 基本的な統計量（件数、合計、平均など）を表示
print("\n--- データの統計情報 ---")
display(df.describe())

--- データの先頭5行 ---


,伝票番号,日付,勘定科目コード,借方金額,貸方金額,摘要
0,1,2026-04-01,100501,100000,100000,備品
1,2,2026-04-01,100601,200000,200000,買掛金支払
2,3,2026-04-01,100701,300000,300000,売掛金入金
3,4,2026-04-02,100802,400000,400000,減価償却・設備
4,5,2026-04-02,200101,500000,500000,売上



--- データの統計情報 ---


,伝票番号,日付,勘定科目コード,借方金額,貸方金額
count,5.000000,5,5.000000,5.000000,5.000000
mean,3.000000,2026-04-01 09:36:00,120541.200000,300000.000000,300000.000000
min,1.000000,2026-04-01 00:00:00,100501.000000,100000.000000,100000.000000
25%,2.000000,2026-04-01 00:00:00,100601.000000,200000.000000,200000.000000
50%,3.000000,2026-04-01 00:00:00,100701.000000,300000.000000,300000.000000
75%,4.000000,2026-04-02 00:00:00,100802.000000,400000.000000,400000.000000
max,5.000000,2026-04-02 00:00:00,200101.000000,500000.000000,500000.000000
std,1.581139,NaN,44475.421642,158113.883008,158113.883008


In [1]:
import pandas as pd

# 1. 取引データ（請求時と支払時のデータ）
# ※ 伝票 FX003 は、入力された損益額が間違っている想定（計算チェック用）
exchange_data = {
    '取引ID': ['FX001', 'FX002', 'FX003'],
    '外貨額_USD': [100.0, 250.0, 150.0],
    '請求時レート': [150.0, 151.0, 149.0],
    '支払時レート': [155.0, 148.0, 152.0],
    '入力済為替損益': [-500, 750, -100] # FX003は本来 -450のはず
}
df_fx = pd.DataFrame(exchange_data)

# 2. 為替差損益の計算ロジック
# 計算式: 外貨額 * (請求時レート - 支払時レート)
# ※ 正数なら「益」、負数なら「損」
df_fx['理論上の為替損益'] = (
    df_fx['外貨額_USD'] * (df_fx['請求時レート'] - df_fx['支払時レート'])
).astype(int)

# 3. 誤差の検知
df_fx['計算誤差'] = df_fx['入力済為替損益'] - df_fx['理論上の為替損益']
error_fx = df_fx[df_fx['計算誤差'] != 0]

print("--- 為替差損益（実現損益）検証レポート ---")
if len(error_fx) > 0:
    print(f"⚠️ 警告: 為替差損益の計算不整合が {len(error_fx)} 件あります。")
    display(error_fx[['取引ID', '外貨額_USD', '理論上の為替損益', '入力済為替損益', '計算誤差']])
else:
    print("✅ すべての為替差損益が正しく計上されています。")

--- 為替差損益（実現損益）検証レポート ---
⚠️ 警告: 為替差損益の計算不整合が 1 件あります。


,取引ID,外貨額_USD,理論上の為替損益,入力済為替損益,計算誤差
2,FX003,150.0,-450,-100,350


In [ ]:
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

def calculate_tax(amount, rate, method=ROUND_HALF_UP):
    """
    正確な端数処理（四捨五入）を行う税額計算関数
    """
    tax = Decimal(str(amount)) * Decimal(str(rate))
    return int(tax.to_integral_value(rounding=method))

# 1. 検証対象の請求書データ（入力済みデータ）
# ※ 3行目は税額が間違っている（1,000円のはずが1,100円になっている）想定
invoice_data = {
    '請求書No': ['INV001', 'INV002', 'INV003', 'INV004'],
    '税抜金額': [10000, 20000, 10000, 5555],
    '適用税率': [0.10, 0.08, 0.10, 0.10],
    '入力済税額': [1000, 1600, 1100, 556]
}
df_inv = pd.DataFrame(invoice_data)

# 2. Pythonロジックで「正しい税額」を再計算
df_inv['理論上の税額'] = df_inv.apply(
    lambda x: calculate_tax(x['税抜金額'], x['適用税率']), axis=1
)

# 3. 誤差（不整合）をチェック
df_inv['誤差'] = df_inv['入力済税額'] - df_inv['理論上の税額']
discrepancy_df = df_inv[df_inv['誤差'] != 0]

print("--- 請求書税額検証レポート ---")
if len(discrepancy_df) > 0:
    print(f"⚠️ 警告: 税額の不整合が {len(discrepancy_df)} 件見つかりました。")
    display(discrepancy_df)
else:
    print("✅ すべての税額計算が適正です。")

--- 請求書税額検証レポート ---
⚠️ 警告: 税額の不整合が 1 件見つかりました。


,請求書No,税抜金額,適用税率,入力済税額,理論上の税額,誤差
2,INV003,10000,0.1,1100,1000,100


In [ ]:
import pandas as pd
from datetime import datetime

# 1. 改正基準日の設定（この日以降は 8%）
REVISION_DATE = datetime(2024, 10, 1)
OLD_RATE = 0.10
NEW_RATE = 0.08

# 2. 検証対象の取引データ（4行目は「10/1以降なのに10%」で計算されているミスを想定）
tx_data = {
    '伝票No': ['V001', 'V002', 'V003', 'V004'],
    '取引日': ['2024-09-25', '2024-09-30', '2024-10-01', '2024-10-05'],
    '税抜金額': [10000, 20000, 30000, 40000],
    '適用された税率': [0.10, 0.10, 0.08, 0.10]  # V004が設定ミス
}
df_tx = pd.DataFrame(tx_data)
df_tx['取引日'] = pd.to_datetime(df_tx['取引日'])

# 3. 理論上（法律上）あるべき税率を判定する関数
def get_expected_rate(row_date):
    return NEW_RATE if row_date >= REVISION_DATE else OLD_RATE

# 4. 検証ロジックの実行
df_tx['本来の税率'] = df_tx['取引日'].apply(get_expected_rate)
df_tx['税率不整合'] = df_tx['適用された税率'] != df_tx['本来の税率']

# 5. 不整合データの抽出
error_tx = df_tx[df_tx['税率不整合'] == True]

print(f"--- 税率改正（基準日: {REVISION_DATE.date()}）対応検証結果 ---")
if len(error_tx) > 0:
    print(f"⚠️ 警告: 適用日の判定ミスと思われるデータが {len(error_tx)} 件あります。")
    display(error_tx[['伝票No', '取引日', '適用された税率', '本来の税率']])
else:
    print("✅ すべての取引で正しい税率が適用されています。")

--- 税率改正（基準日: 2024-10-01）対応検証結果 ---
⚠️ 警告: 適用日の判定ミスと思われるデータが 1 件あります。


,伝票No,取引日,適用された税率,本来の税率
3,V004,2024-10-05,0.1,0.08


In [ ]:
import pandas as pd

# 1. 商品マスターの設定（品目ごとの正しい税率）
item_master = {
    '品目': ['りんご', 'お茶', '洗剤', 'ノート'],
    '正しい税率': [0.08, 0.08, 0.10, 0.10]
}
df_master = pd.DataFrame(item_master)

# 2. 実際に届いた販売伝票データ
# ※「お茶」が 0.10 で計算されているミスを想定
sales_data = {
    '伝票No': ['S001', 'S002', 'S003', 'S004'],
    '品目': ['りんご', 'お茶', '洗剤', 'ノート'],
    '適用税率': [0.08, 0.10, 0.10, 0.10]  # S002がミス
}
df_sales = pd.DataFrame(sales_data)

# 3. マスターデータを結合（Merge）して、本来の税率を紐付ける
df_check = pd.merge(df_sales, df_master, on='品目', how='left')

# 4. 不整合（Mismatch）の検知
# ここでも「mismatch」という名前を自分で決めて使います
mismatch_data = df_check[df_check['適用税率'] != df_check['正しい税率']]

print("--- 軽減税率・標準税率 適用チェック ---")
if len(mismatch_data) > 0:
    print(f"⚠️ 警告: 品目マスターと異なる税率が適用されている伝票が {len(mismatch_data)} 件あります。")
    display(mismatch_data[['伝票No', '品目', '適用税率', '正しい税率']])
else:
    print("✅ すべての品目に対して正しい税率が適用されています。")

--- 軽減税率・標準税率 適用チェック ---
⚠️ 警告: 品目マスターと異なる税率が適用されている伝票が 1 件あります。


,伝票No,品目,適用税率,正しい税率
1,S002,お茶,0.1,0.08


In [ ]:
import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, 'abcd', 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# 「金額に数字ではない文字」が入っている場合は警告を出す処理
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-05', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['取引日', '金額'],
    right_on=['仕訳日', '金額'],
    how='inner'
)

# --- エラーハンドリング処理 ---
# 金額を数値に変換。変換できない文字は強制的に「NaN（欠損値）」にする
df_bank['金額'] = pd.to_numeric(df_bank['金額'], errors='coerce')

# NaN（数字じゃなかった行）を探して警告を出す
invalid_rows = df_bank[df_bank['金額'].isnull()]

print("--- データ不正チェック ---")
if len(invalid_rows) > 0:
    print(f"⚠️ 警告: 数字ではないデータが {len(invalid_rows)} 件見つかりました。")
    display(invalid_rows)
else:
    print("✅ 全ての金額データが正常な数値です。")

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])
display(unmatched_bank)

--- データ不正チェック ---
⚠️ 警告: 数字ではないデータが 1 件見つかりました。


,取引日,金額,銀行参照番号
2,2024-04-10,NaN,BK003


✅ 照合成功（日付と金額が一致）:


,取引日,金額,銀行参照番号,伝票番号
0,2024-04-01,50000,BK001,V-001
1,2024-04-05,120000,BK002,V-002


,取引日,金額,銀行参照番号
2,2024-04-10,NaN,BK003
3,2024-04-15,45000.0,BK004


In [ ]:
import pandas as pd

def calculate_straight_line_depreciation(acquisition_cost, salvage_value, useful_life_years):
    """
    定額法の減価償却スケジュールを計算する関数
    """
    # 年間の償却額 = (取得価額 - 残存価額) / 耐用年数
    annual_depreciation = (acquisition_cost - salvage_value) / useful_life_years

    schedule = []
    current_book_value = acquisition_cost

    for year in range(1, useful_life_years + 1):
        # 最終年の調整（残存価額を下回らないようにする）
        if year == useful_life_years:
            depreciation_amount = current_book_value - salvage_value
        else:
            depreciation_amount = annual_depreciation

        current_book_value -= depreciation_amount

        schedule.append({
            "年": year,
            "期首帳簿価額": round(current_book_value + depreciation_amount),
            "償却額": round(depreciation_amount),
            "期末帳簿価額": round(current_book_value)
        })

    return pd.DataFrame(schedule)

# --- テスト用の設定値 ---
cost = 1200000    # 取得価額（120万円）
salvage = 10000   # 残存価額（1万円）
life = 5          # 耐用年数（5年）

# 実行
depreciation_df = calculate_straight_line_depreciation(cost, salvage, life)

print(f"取得価額: {cost:,}円 / 耐用年数: {life}年 の減価償却スケジュール")
display(depreciation_df)

# Excelとして保存したい場合（任意）
# depreciation_df.to_excel("depreciation_schedule.xlsx", index=False)

取得価額: 1,200,000円 / 耐用年数: 5年 の減価償却スケジュール


,年,期首帳簿価額,償却額,期末帳簿価額
0,1,1200000,238000,962000
1,2,962000,238000,724000
2,3,724000,238000,486000
3,4,486000,238000,248000
4,5,248000,238000,10000


In [ ]:
import pandas as pd

# 1. 為替レートマスタ（日付ごとの1ドルの円相場）
exchange_rates = {
    '日付': ['2024-04-01', '2024-04-02', '2024-04-03'],
    'レート': [151.20, 150.85, 151.50]
}
df_rates = pd.DataFrame(exchange_rates)

# 2. 外貨建て取引データ（USD）
# ※ 2行目は「150.85」で計算すべきところが間違っている想定
foreign_tx = {
    '伝票No': ['FX001', 'FX002', 'FX003'],
    '日付': ['2024-04-01', '2024-04-02', '2024-04-03'],
    '外貨額_USD': [100.00, 200.00, 150.00],
    '入力済邦貨額_JPY': [15120, 30100, 22725]
}
df_tx = pd.DataFrame(foreign_tx)

# 3. レートを紐付けて理論上の邦貨額を計算
df_val = pd.merge(df_tx, df_rates, on='日付', how='left')
df_val['理論上の邦貨額'] = (df_val['外貨額_USD'] * df_val['レート']).round(0).astype(int)

# 4. 誤差の抽出
df_val['換算誤差'] = df_val['入力済邦貨額_JPY'] - df_val['理論上の邦貨額']
fx_error = df_val[df_val['換算誤差'] != 0]

print("--- 外貨換算（USD -> JPY）整合性チェック ---")
if len(fx_error) > 0:
    print(f"⚠️ 警告: 為替換算に誤りがある可能性が高い伝票が {len(fx_error)} 件あります。")
    display(fx_error[['伝票No', '日付', '外貨額_USD', 'レート', '理論上の邦貨額', '入力済邦貨額_JPY']])
else:
    print("✅ すべての外貨取引が適切なレートで換算されています。")

--- 外貨換算（USD -> JPY）整合性チェック ---
⚠️ 警告: 為替換算に誤りがある可能性が高い伝票が 1 件あります。


,伝票No,日付,外貨額_USD,レート,理論上の邦貨額,入力済邦貨額_JPY
1,FX002,2024-04-02,200.0,150.85,30170,30100


In [ ]:
import pandas as pd
import math

def calculate_monthly_depreciation(acquisition_cost, salvage_value, useful_life_years):
    """
    定額法に基づき、月次の償却スケジュールを生成する関数
    """
    # 総償却月数
    total_months = useful_life_years * 12
    # 総償却対象額
    total_depreciable_amount = acquisition_cost - salvage_value
    # 月次の標準償却額（端数は切り捨て、最終月で調整）
    monthly_depreciation = math.floor(total_depreciable_amount / total_months)

    schedule = []
    accumulated_depreciation = 0 # 累計償却額

    for month in range(1, total_months + 1):
        # 最終月の判定（残りの償却対象額をすべて計上して端数を調整）
        if month == total_months:
            depreciation_amount = total_depreciable_amount - accumulated_depreciation
        else:
            depreciation_amount = monthly_depreciation

        accumulated_depreciation += depreciation_amount
        current_book_value = acquisition_cost - accumulated_depreciation

        schedule.append({
            "月": month,
            "当月償却額": int(depreciation_amount),
            "累計償却額": int(accumulated_depreciation),
            "期末帳簿価額": int(current_book_value)
        })

    return pd.DataFrame(schedule)

# --- テスト実行 ---
cost = 1200000
salvage = 1
years = 5

monthly_df = calculate_monthly_depreciation(cost, salvage, years)

print(f"【月次】減価償却シミュレーション ({years}年 = {years*12}ヶ月)")
# 全件表示すると長いので、最初と最後だけ表示
display(monthly_df.head(12)) # 1年目
print("...")
display(monthly_df.tail(3))  # 最終月付近

【月次】減価償却シミュレーション (5年 = 60ヶ月)


,月,当月償却額,累計償却額,期末帳簿価額
0,1,19999,19999,1180001
1,2,19999,39998,1160002
2,3,19999,59997,1140003
3,4,19999,79996,1120004
4,5,19999,99995,1100005
5,6,19999,119994,1080006
6,7,19999,139993,1060007
7,8,19999,159992,1040008
8,9,19999,179991,1020009
9,10,19999,199990,1000010


...


,月,当月償却額,累計償却額,期末帳簿価額
57,58,19999,1159942,40058
58,59,19999,1179941,20059
59,60,20058,1199999,1


In [ ]:
import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, 3000, 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-05', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['取引日', '金額'],
    right_on=['仕訳日', '金額'],
    how='inner'
)

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])

print("\n⚠️ 銀行明細のみに存在（D365未入力の可能性）:")
display(unmatched_bank)

✅ 照合成功（日付と金額が一致）:


,取引日,金額,銀行参照番号,伝票番号
0,2024-04-01,50000,BK001,V-001
1,2024-04-05,120000,BK002,V-002



⚠️ 銀行明細のみに存在（D365未入力の可能性）:


,取引日,金額,銀行参照番号
2,2024-04-10,3000,BK003
3,2024-04-15,45000,BK004


In [ ]:
import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, 3000, 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# 「1日のズレ」を許容する照合
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-06', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['金額'],
    right_on=['金額'],
    how='inner'
)

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])

print("\n⚠️ 銀行明細のみに存在（D365未入力の可能性）:")
display(unmatched_bank)

✅ 照合成功（日付と金額が一致）:


,取引日,金額,銀行参照番号,伝票番号
0,2024-04-01,50000,BK001,V-001
1,2024-04-05,120000,BK002,V-002



⚠️ 銀行明細のみに存在（D365未入力の可能性）:


,取引日,金額,銀行参照番号
2,2024-04-10,3000,BK003
3,2024-04-15,45000,BK004


In [ ]:
import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, 3000, 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# 「1日のズレ」を許容する照合
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-06', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['金額'],
    right_on=['金額'],
    how='inner'
)

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])

print("\n⚠️ 銀行明細のみに存在（D365未入力の可能性）:")
display(unmatched_bank)


import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, abcd, 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# 「金額に数字ではない文字」が入っている場合は警告を出す処理
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-05', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['取引日', '金額'],
    right_on=['仕訳日', '金額'],
    how='inner'
)

# --- エラーハンドリング処理 ---
# 金額を数値に変換。変換できない文字は強制的に「NaN（欠損値）」にする
df_bank['金額'] = pd.to_numeric(df_bank['金額'], errors='coerce')

# NaN（数字じゃなかった行）を探して警告を出す
invalid_rows = df_bank[df_bank['金額'].isnull()]

print("--- データ不正チェック ---")
if len(invalid_rows) > 0:
    print(f"⚠️ 警告: 数字ではないデータが {len(invalid_rows)} 件見つかりました。")
    display(invalid_rows)
else:
    print("✅ 全ての金額データが正常な数値です。")

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])
display(unmatched_bank)

print("\n⚠️ 銀行明細のみに存在（D365未入力の可能性）:")
display(unmatched_bank)

✅ 照合成功（日付と金額が一致）:


,取引日,金額,銀行参照番号,伝票番号
0,2024-04-01,50000,BK001,V-001
1,2024-04-05,120000,BK002,V-002



⚠️ 銀行明細のみに存在（D365未入力の可能性）:


,取引日,金額,銀行参照番号
2,2024-04-10,3000,BK003
3,2024-04-15,45000,BK004


NameError: name 'abcd' is not defined

In [ ]:
# テストデータ（金額に「あいうえお」という不正な値が混じっている想定）
data = {
    '取引日': ['2024-04-01', '2024-04-02', '2024-04-03'],
    '金額': [50000, 'あいうえお', 120000] # 文字列が混入
}
df_bank = pd.DataFrame(data)

# --- エラーハンドリングの核心 ---
# 金額を数値に変換。変換できない文字は強制的に「NaN（欠損値）」にする
df_bank['金額'] = pd.to_numeric(df_bank['金額'], errors='coerce')

# NaN（数字じゃなかった行）を探して警告を出す
invalid_rows = df_bank[df_bank['金額'].isnull()]

print("--- データ不正チェック ---")
if len(invalid_rows) > 0:
    print(f"⚠️ 警告: 数字ではないデータが {len(invalid_rows)} 件見つかりました。")
    display(invalid_rows)
else:
    print("✅ 全ての金額データが正常な数値です。")

# 正常なデータだけで処理を続ける
df_valid = df_bank.dropna(subset=['金額'])

--- データ不正チェック ---
⚠️ 警告: 数字ではないデータが 1 件見つかりました。


,取引日,金額
1,2024-04-02,NaN


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print("Current working directory files:")
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Current working directory files:
./journal_test.xlsx
./test_data_2.xlsx
./test_data.xlsx
./.config/active_config
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/gce
./.config/.last_opt_in_prompt.yaml
./.config/.last_survey_prompt.yaml
./.config/config_sentinel
./.config/default_configs.db
./.config/.last_update_check.json
./.config/logs/2026.03.30/13.29.11.295522.log
./.config/logs/2026.03.30/13.28.36.228080.log
./.config/logs/2026.03.30/13.29.00.305433.log
./.config/logs/2026.03.30/13.29.12.938774.log
./.config/logs/2026.03.30/13.29.24.176304.log
./.config/logs/2026.03.30/13.29.24.988708.log
./.config/configurations/config_default
./sample_data/README.md
./sample_data/anscombe.json
./sample_data/mnist_train_small.csv
./sample_data/mnist_test.csv
./sample_data/california_housing_test.csv
./sample_data/california_housing_train.csv


In [ ]:
import pandas as pd

# Excelファイルを読み込む
df = pd.read_excel('test_data_2.xlsx')

# 現在読み込んでいるデータの列名をすべて表示する
print("--- 現在のExcelの列名一覧 ---")
print(df.columns.tolist())

# 列のデータ型や欠損値もまとめて確認する
print("\n--- データの詳細情報 ---")

# 1. 貸借一致チェック（会計コンサルの基本！）
debit_total = df['借方金額'].sum()
credit_total = df['貸方金額'].sum()

print(f"--- 貸借チェック結果 ---")
if debit_total == credit_total:
    print(f"✅ OK: 貸借が一致しています（合計: {debit_total}）")
else:
    print(f"❌ NG: 貸借が不一致です（借方: {debit_total}, 貸方: {credit_total}）")

# 2. 必須項目の空欄（欠損値）チェック
print("\n--- 必須項目チェック ---")
missing_accounts = df['勘定科目コード'].isnull().sum()
if missing_accounts > 0:
    print(f"⚠️ 警告: 勘定科目コードが入力されていない行が {missing_accounts} 件あります。")
else:
    print("✅ OK: すべての行に勘定科目コードが入っています。")

--- 現在のExcelの列名一覧 ---
['伝票番号', '日付', '勘定科目コード', '借方金額', '貸方金額', '摘要']

--- データの詳細情報 ---
--- 貸借チェック結果 ---
✅ OK: 貸借が一致しています（合計: 1500000）

--- 必須項目チェック ---
✅ OK: すべての行に勘定科目コードが入っています。
